# Notebook 09 — Agentes y el patrón ReAct

**Semana:** 9  
**Bloque:** 5.1 — Agentes autónomos  
**Tiempo estimado:** 4-5 horas

## Qué vas a construir

- Un bucle de agente genérico que funciona con cualquier conjunto de herramientas
- Un agente de investigación con 4 herramientas reales (Wikipedia, calculadora, fecha/hora, búsqueda de noticias simulada)
- Trazabilidad completa: cada paso del agente queda registrado
- **Reto final:** agente de análisis financiero que responde preguntas sobre empresas usando múltiples fuentes

## Prerrequisitos

Semanas 2 y 3 completadas (function calling, manejo de errores).

```bash
pip install openai anthropic requests python-dotenv
```

---
## Celda 0 — Setup

In [ ]:
import os
import json
import math
import uuid
import time
import datetime
import operator
import requests
from dataclasses import dataclass, field, asdict
from typing import Any, Callable
from dotenv import load_dotenv
from openai import OpenAI
import anthropic

load_dotenv()

cliente_openai    = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
cliente_anthropic = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

MODELO_OPENAI    = "gpt-4o-mini"
MODELO_ANTHROPIC = "claude-haiku-4-5-20251001"

print("Setup completado.")
print(f"OpenAI:    {MODELO_OPENAI}")
print(f"Anthropic: {MODELO_ANTHROPIC}")

---
## Celda 1 — Las herramientas del agente

Define 4 herramientas reales.
Cada herramienta es una función Python + su esquema JSON Schema para el LLM.

In [ ]:
# ── Herramienta 1: Wikipedia ──────────────────────────────────────────────────

def buscar_wikipedia(query: str, sentences: int = 3) -> str:
    """
    Busca información factual en Wikipedia.
    Devuelve un extracto del artículo más relevante.
    """
    url    = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query", "format": "json", "list": "search",
        "srsearch": query, "srlimit": 1
    }
    try:
        resultados = requests.get(url, params=params, timeout=8).json()
        if not resultados["query"]["search"]:
            return f"No se encontraron resultados para '{query}'."

        titulo  = resultados["query"]["search"][0]["title"]
        params2 = {
            "action": "query", "format": "json",
            "titles": titulo, "prop": "extracts",
            "exsentences": sentences, "exintro": True, "explaintext": True
        }
        pagina = requests.get(url, params=params2, timeout=8).json()
        pages  = pagina["query"]["pages"]
        texto  = list(pages.values())[0].get("extract", "Sin contenido.")
        return f"[Wikipedia: {titulo}]\n{texto}"
    except Exception as e:
        return f"Error al buscar en Wikipedia: {e}"


# ── Herramienta 2: Calculadora ────────────────────────────────────────────────

def calcular(expresion: str) -> str:
    """
    Evalúa una expresión matemática de forma segura.
    Soporta +, -, *, /, **, sqrt, log, abs, round.
    """
    permitidos = {
        "__builtins__": {},
        "abs": abs, "round": round,
        "sqrt": math.sqrt, "log": math.log, "log10": math.log10,
        "sin": math.sin, "cos": math.cos, "tan": math.tan,
        "pi": math.pi, "e": math.e,
    }
    try:
        resultado = eval(expresion, permitidos)
        return f"{expresion} = {resultado}"
    except ZeroDivisionError:
        return "Error: división por cero."
    except Exception as e:
        return f"Error evaluando '{expresion}': {e}"


# ── Herramienta 3: Fecha y hora ───────────────────────────────────────────────

def obtener_fecha_hora(zona_horaria: str = "UTC") -> str:
    """
    Devuelve la fecha y hora actual en formato ISO.
    """
    ahora = datetime.datetime.utcnow()
    return json.dumps({
        "datetime_utc": ahora.isoformat(),
        "date":         ahora.strftime("%Y-%m-%d"),
        "time":         ahora.strftime("%H:%M:%S"),
        "weekday":      ahora.strftime("%A"),
        "timezone_note": f"Requested: {zona_horaria} (returned UTC)"
    })


# ── Herramienta 4: Noticias simuladas ────────────────────────────────────────
# En producción esto llamaría a NewsAPI, Bing News o similar.
# Aquí simulamos con datos fijos para no necesitar otra API key.

NOTICIAS_SIMULADAS = {
    "python": [
        "Python 3.13 released with JIT compiler improvements (Oct 2024)",
        "Python ranked #1 language in TIOBE index for 4th consecutive year",
    ],
    "ai": [
        "OpenAI releases GPT-4o with multimodal capabilities (May 2024)",
        "Anthropic raises $2.75B to develop Claude AI models",
        "EU AI Act enters into force, first comprehensive AI regulation",
    ],
    "fastapi": [
        "FastAPI 0.115 released with improved async support",
        "FastAPI surpasses 70k GitHub stars, most starred Python API framework",
    ],
    "default": [
        "No recent news found for this topic in the simulated database.",
    ]
}

def buscar_noticias(tema: str, max_resultados: int = 3) -> str:
    """
    Busca noticias recientes sobre un tema.
    SIMULADO: en producción conecta a NewsAPI o similar.
    """
    clave    = next((k for k in NOTICIAS_SIMULADAS if k in tema.lower()), "default")
    noticias = NOTICIAS_SIMULADAS[clave][:max_resultados]
    return f"[Noticias sobre '{tema}']\n" + "\n".join(f"- {n}" for n in noticias)


# ── Registro de herramientas ──────────────────────────────────────────────────

FUNCIONES_DISPONIBLES: dict[str, Callable] = {
    "buscar_wikipedia": buscar_wikipedia,
    "calcular":         calcular,
    "obtener_fecha_hora": obtener_fecha_hora,
    "buscar_noticias":  buscar_noticias,
}

# Esquemas JSON Schema para OpenAI
HERRAMIENTAS_OPENAI = [
    {
        "type": "function",
        "function": {
            "name": "buscar_wikipedia",
            "description": (
                "Search Wikipedia for factual background information, definitions, "
                "historical data, or explanations. Use for stable facts, NOT for "
                "real-time data like prices, weather, or breaking news."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query":     {"type": "string", "description": "Search term or question"},
                    "sentences": {"type": "integer", "description": "Number of sentences (1-10)", "default": 3}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calcular",
            "description": (
                "Evaluate a mathematical expression precisely. Supports +, -, *, /, **, "
                "sqrt(), log(), sin(), cos(), abs(), round(). Always use this instead "
                "of computing in your head."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "expresion": {"type": "string", "description": "Valid Python math expression, e.g. 'sqrt(144) * 2.5'"}
                },
                "required": ["expresion"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "obtener_fecha_hora",
            "description": "Get the current date and time. Use whenever the task requires knowing today's date.",
            "parameters": {
                "type": "object",
                "properties": {
                    "zona_horaria": {"type": "string", "description": "Timezone name, e.g. 'Europe/Madrid'", "default": "UTC"}
                },
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "buscar_noticias",
            "description": (
                "Search for recent news articles about a topic. Use for current events, "
                "product releases, company updates, or anything that changes frequently. "
                "Do NOT use for historical or encyclopedic information (use Wikipedia instead)."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "tema":           {"type": "string",  "description": "Topic to search for"},
                    "max_resultados": {"type": "integer", "description": "Max results (1-5)", "default": 3}
                },
                "required": ["tema"]
            }
        }
    }
]

# ── Test rápido de las herramientas ──────────────────────────────────────────
print("Test de herramientas:")
print(buscar_wikipedia("FastAPI", sentences=2))
print(calcular("sqrt(144) + 2**8"))
print(obtener_fecha_hora())
print(buscar_noticias("AI", max_resultados=2))

---
## Celda 2 — El bucle del agente (OpenAI)

Implementa el bucle ReAct completo con trazabilidad.
El agente llama al LLM, ejecuta las herramientas que pide, y repite hasta que responde directamente.

In [ ]:
# ── Estructuras de trazabilidad ───────────────────────────────────────────────

@dataclass
class PasoAgente:
    paso:        int
    tipo:        str          # "tool_call" | "parallel_tool_calls" | "final_answer" | "error"
    herramienta: str | None
    argumentos:  dict | None
    resultado:   str | None
    latencia_ms: float

@dataclass
class TrazaAgente:
    id:             str
    pregunta:       str
    respuesta:      str | None
    pasos:          list[PasoAgente] = field(default_factory=list)
    tokens_totales: int   = 0
    completado:     bool  = False
    razon_parada:   str   = ""

    def coste_usd(self) -> float:
        return self.tokens_totales * 0.00000015   # gpt-4o-mini aprox.

    def resumen(self) -> None:
        n_herramientas = sum(1 for p in self.pasos if p.tipo in ("tool_call", "parallel_tool_calls"))
        print(f"\n{'='*55}")
        print(f"ID:            {self.id}")
        print(f"Completado:    {self.completado} ({self.razon_parada})")
        print(f"Pasos totales: {len(self.pasos)} ({n_herramientas} llamadas a herramientas)")
        print(f"Tokens:        {self.tokens_totales}")
        print(f"Coste aprox.:  ${self.coste_usd():.6f}")
        print(f"{'='*55}")
        for p in self.pasos:
            if p.tipo in ("tool_call", "parallel_tool_calls"):
                args_str = json.dumps(p.argumentos, ensure_ascii=False)[:60]
                res_str  = str(p.resultado)[:60] if p.resultado else "—"
                print(f"  [{p.paso}] {p.herramienta}({args_str}...)")
                print(f"       → {res_str}...")
            else:
                print(f"  [{p.paso}] {p.tipo}")


# ── Ejecutor de herramientas ──────────────────────────────────────────────────

def ejecutar_herramienta(nombre: str, argumentos: dict) -> str:
    """Llama a la función Python correspondiente y devuelve el resultado como string."""
    if nombre not in FUNCIONES_DISPONIBLES:
        return f"Error: herramienta '{nombre}' no existe."
    try:
        return str(FUNCIONES_DISPONIBLES[nombre](**argumentos))
    except TypeError as e:
        return f"Error en argumentos de '{nombre}': {e}"
    except Exception as e:
        return f"Error ejecutando '{nombre}': {e}"


# ── Bucle del agente (OpenAI) ─────────────────────────────────────────────────

SYSTEM_AGENTE = """You are a research assistant with access to tools.
Always use tools to verify facts rather than relying on your training data.
When you have all the information needed, provide a clear, concise final answer.
Cite your sources (tool names and results) in the answer."""


def ejecutar_agente_openai(
    pregunta: str,
    herramientas: list = HERRAMIENTAS_OPENAI,
    max_pasos: int = 8,
    verbose: bool = True
) -> TrazaAgente:
    """
    Bucle ReAct con OpenAI.
    El LLM decide en cada paso si usar herramientas o responder directamente.
    """
    traza    = TrazaAgente(id=str(uuid.uuid4())[:8], pregunta=pregunta, respuesta=None)
    mensajes = [
        {"role": "system",  "content": SYSTEM_AGENTE},
        {"role": "user",    "content": pregunta}
    ]

    if verbose:
        print(f"\nAgente iniciado: '{pregunta[:60]}...'" if len(pregunta) > 60 else f"\nAgente iniciado: '{pregunta}'")

    for paso in range(1, max_pasos + 1):
        t_inicio = time.perf_counter()

        respuesta_llm = cliente_openai.chat.completions.create(
            model=MODELO_OPENAI,
            messages=mensajes,
            tools=herramientas,
            tool_choice="auto"
        )

        latencia_ms       = (time.perf_counter() - t_inicio) * 1000
        mensaje_asistente = respuesta_llm.choices[0].message
        traza.tokens_totales += respuesta_llm.usage.total_tokens

        # ── Sin herramientas: respuesta final ─────────────────────────────────
        if not mensaje_asistente.tool_calls:
            traza.pasos.append(PasoAgente(
                paso=paso, tipo="final_answer",
                herramienta=None, argumentos=None,
                resultado=mensaje_asistente.content, latencia_ms=round(latencia_ms, 1)
            ))
            traza.respuesta   = mensaje_asistente.content
            traza.completado  = True
            traza.razon_parada = "respuesta_directa"
            if verbose:
                print(f"  [Paso {paso}] Respuesta final generada.")
            break

        # ── Con herramientas: ejecuta cada tool call ──────────────────────────
        n_calls = len(mensaje_asistente.tool_calls)
        if verbose:
            herrs = ", ".join(tc.function.name for tc in mensaje_asistente.tool_calls)
            print(f"  [Paso {paso}] Herramientas solicitadas ({n_calls}): {herrs}")

        # Añade el mensaje del asistente al historial
        mensajes.append(mensaje_asistente)

        # Ejecuta todas las herramientas solicitadas
        for tool_call in mensaje_asistente.tool_calls:
            t_herramienta = time.perf_counter()
            nombre        = tool_call.function.name
            try:
                argumentos = json.loads(tool_call.function.arguments)
            except json.JSONDecodeError:
                argumentos = {}

            resultado    = ejecutar_herramienta(nombre, argumentos)
            lat_herr_ms  = (time.perf_counter() - t_herramienta) * 1000

            tipo_paso = "parallel_tool_calls" if n_calls > 1 else "tool_call"
            traza.pasos.append(PasoAgente(
                paso=paso, tipo=tipo_paso,
                herramienta=nombre, argumentos=argumentos,
                resultado=resultado[:200], latencia_ms=round(lat_herr_ms, 1)
            ))

            # Añade el resultado al historial como mensaje de herramienta
            mensajes.append({
                "role":         "tool",
                "tool_call_id": tool_call.id,
                "content":      resultado
            })

            if verbose:
                print(f"    → {nombre}: {resultado[:80]}...")

    else:
        # Agotamos max_pasos sin respuesta directa
        traza.razon_parada = "limite_pasos"
        traza.respuesta    = "El agente superó el límite de pasos sin llegar a una respuesta."

    return traza


# ── Prueba 1: tarea simple (una herramienta) ──────────────────────────────────
t1 = ejecutar_agente_openai("What is the capital of France and what is its population?")
t1.resumen()
print(f"\nRespuesta: {t1.respuesta}")

---
## Celda 3 — Tareas que requieren múltiples herramientas

Prueba el agente con tareas que requieren combinar información de varias fuentes.
Observa si el LLM usa parallel tool calls o llamadas secuenciales.

In [ ]:
PREGUNTAS_COMPLEJAS = [
    # Requiere Wikipedia + calculadora
    "The speed of light is approximately 299,792 km/s. How many seconds does it take "
    "light to travel from the Sun to Earth? (the Sun-Earth distance is about 149.6 million km)",

    # Requiere Wikipedia + noticias
    "What is FastAPI and what are the most recent updates about it?",

    # Requiere fecha + Wikipedia
    "How old is Python today? (it was first released on February 20, 1991)",
]

for i, pregunta in enumerate(PREGUNTAS_COMPLEJAS, 1):
    print(f"\n{'='*60}")
    print(f"TAREA {i}")
    print(f"{'='*60}")
    traza = ejecutar_agente_openai(pregunta, verbose=True)
    traza.resumen()
    print(f"\nRespuesta final:\n{traza.respuesta}")

**Observaciones** (edita esta celda):

- ¿En qué tareas usó parallel tool calls (varias herramientas en el mismo paso)?
- ¿Hubo alguna tarea donde el agente usó más pasos de los necesarios?
- ¿Alguna respuesta contiene información que el agente debería haber verificado pero no verificó?

---
## Celda 4 — El mismo agente con Anthropic (Claude)

Implementa el mismo bucle ReAct para Anthropic.
La diferencia clave: los resultados de herramientas van en mensajes `role: user`, no `role: tool`.

In [ ]:
# Convierte los esquemas de OpenAI al formato de Anthropic
def openai_tools_to_anthropic(herramientas_openai: list) -> list:
    """Adapta el formato de herramientas de OpenAI al de Anthropic."""
    resultado = []
    for h in herramientas_openai:
        fn = h["function"]
        resultado.append({
            "name":         fn["name"],
            "description":  fn["description"],
            "input_schema": fn["parameters"]
        })
    return resultado

HERRAMIENTAS_ANTHROPIC = openai_tools_to_anthropic(HERRAMIENTAS_OPENAI)


def ejecutar_agente_anthropic(
    pregunta: str,
    herramientas: list = HERRAMIENTAS_ANTHROPIC,
    max_pasos: int = 8,
    verbose: bool = True
) -> TrazaAgente:
    """
    Bucle ReAct con Anthropic Claude.
    Diferencias vs OpenAI:
    - stop_reason="tool_use" indica que el LLM quiere usar herramientas
    - El historial usa role:user con tipo tool_result para los resultados
    """
    traza    = TrazaAgente(id=str(uuid.uuid4())[:8], pregunta=pregunta, respuesta=None)
    mensajes = [{"role": "user", "content": pregunta}]

    if verbose:
        print(f"\nAgente Claude: '{pregunta[:60]}...'" if len(pregunta) > 60 else f"\nAgente Claude: '{pregunta}'")

    for paso in range(1, max_pasos + 1):
        t_inicio = time.perf_counter()

        respuesta_llm = cliente_anthropic.messages.create(
            model=MODELO_ANTHROPIC,
            max_tokens=1024,
            system=SYSTEM_AGENTE,
            messages=mensajes,
            tools=herramientas
        )

        latencia_ms = (time.perf_counter() - t_inicio) * 1000
        uso         = respuesta_llm.usage
        traza.tokens_totales += uso.input_tokens + uso.output_tokens

        # ── Sin herramientas: respuesta final ─────────────────────────────────
        if respuesta_llm.stop_reason == "end_turn":
            texto_final = "".join(
                b.text for b in respuesta_llm.content if hasattr(b, "text")
            )
            traza.pasos.append(PasoAgente(
                paso=paso, tipo="final_answer",
                herramienta=None, argumentos=None,
                resultado=texto_final, latencia_ms=round(latencia_ms, 1)
            ))
            traza.respuesta    = texto_final
            traza.completado   = True
            traza.razon_parada = "respuesta_directa"
            if verbose:
                print(f"  [Paso {paso}] Respuesta final generada.")
            break

        # ── Con herramientas: stop_reason == "tool_use" ───────────────────────
        tool_uses = [b for b in respuesta_llm.content if b.type == "tool_use"]
        n_calls   = len(tool_uses)

        if verbose:
            herrs = ", ".join(tu.name for tu in tool_uses)
            print(f"  [Paso {paso}] Herramientas ({n_calls}): {herrs}")

        # Añade el mensaje del asistente al historial
        mensajes.append({"role": "assistant", "content": respuesta_llm.content})

        # Ejecuta las herramientas y construye la respuesta como role:user
        resultados_herramientas = []
        for tool_use in tool_uses:
            t_herr       = time.perf_counter()
            nombre       = tool_use.name
            argumentos   = tool_use.input
            resultado    = ejecutar_herramienta(nombre, argumentos)
            lat_herr_ms  = (time.perf_counter() - t_herr) * 1000

            tipo_paso = "parallel_tool_calls" if n_calls > 1 else "tool_call"
            traza.pasos.append(PasoAgente(
                paso=paso, tipo=tipo_paso,
                herramienta=nombre, argumentos=argumentos,
                resultado=resultado[:200], latencia_ms=round(lat_herr_ms, 1)
            ))

            resultados_herramientas.append({
                "type":        "tool_result",
                "tool_use_id": tool_use.id,
                "content":     resultado
            })

            if verbose:
                print(f"    → {nombre}: {resultado[:80]}...")

        # En Anthropic, los resultados van en un mensaje role:user
        mensajes.append({"role": "user", "content": resultados_herramientas})

    else:
        traza.razon_parada = "limite_pasos"
        traza.respuesta    = "El agente superó el límite de pasos."

    return traza


# ── Compara OpenAI vs Anthropic en la misma tarea ─────────────────────────────
TAREA_COMPARATIVA = (
    "What is Python's current version, what are the most recent news about it, "
    "and how many years has Python existed? (it was first released in February 1991)"
)

print("\nOpenAI:")
traza_oai = ejecutar_agente_openai(TAREA_COMPARATIVA, verbose=True)

print("\nAnthropc:")
traza_ant = ejecutar_agente_anthropic(TAREA_COMPARATIVA, verbose=True)

# Tabla comparativa
print("\n" + "=" * 52)
print(f"{'MÉTRICA':<25} {'OPENAI':>12} {'ANTHROPIC':>12}")
print("=" * 52)
for label, v_oai, v_ant in [
    ("Pasos totales",  len(traza_oai.pasos), len(traza_ant.pasos)),
    ("Tokens totales", traza_oai.tokens_totales, traza_ant.tokens_totales),
    ("Completado",     traza_oai.completado, traza_ant.completado),
]:
    print(f"{label:<25} {str(v_oai):>12} {str(v_ant):>12}")
print("=" * 52)

---
## Reto final — Agente de análisis de tecnología

Construye un agente especializado que responde preguntas sobre una tecnología
combinando información enciclopédica, noticias recientes y cálculos.

Lee el esqueleto y completa los `# TODO`.

In [ ]:
# ── Herramienta adicional: comparador de tecnologías ─────────────────────────
# Esta herramienta combina Wikipedia + estructura de comparación

def comparar_tecnologias(tecnologia_a: str, tecnologia_b: str, aspecto: str = "general") -> str:
    """
    Busca información sobre dos tecnologías y devuelve datos para compararlas.
    """
    info_a = buscar_wikipedia(tecnologia_a, sentences=2)
    info_b = buscar_wikipedia(tecnologia_b, sentences=2)
    return json.dumps({
        "tecnologia_a": {"nombre": tecnologia_a, "info": info_a},
        "tecnologia_b": {"nombre": tecnologia_b, "info": info_b},
        "aspecto":      aspecto,
        "nota":         "Use this data to compare the two technologies based on the specified aspect."
    }, ensure_ascii=False)


HERRAMIENTAS_EXTENDIDAS_OPENAI = HERRAMIENTAS_OPENAI + [
    {
        "type": "function",
        "function": {
            "name": "comparar_tecnologias",
            "description": (
                "Look up and compare two technologies side by side on a specific aspect. "
                "Use this when asked to compare, contrast, or choose between two technologies."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "tecnologia_a": {"type": "string", "description": "First technology name"},
                    "tecnologia_b": {"type": "string", "description": "Second technology name"},
                    "aspecto":      {"type": "string", "description": "Aspect to compare (performance, ecosystem, use cases, etc.)"}
                },
                "required": ["tecnologia_a", "tecnologia_b"]
            }
        }
    }
]

FUNCIONES_DISPONIBLES["comparar_tecnologias"] = comparar_tecnologias


# ── Agente especializado (reto) ───────────────────────────────────────────────

SYSTEM_TECH_ANALYST = """
You are a senior technology analyst specializing in software development tools and frameworks.
Your reports are:
- Fact-based: always verify with tools before stating facts
- Structured: use headers and bullet points
- Actionable: include a concrete recommendation
- Honest about uncertainty: if you can't verify something, say so

Always use the available tools to gather information. Do not rely on training data alone.
"""


def agente_analista(
    pregunta: str,
    max_pasos: int = 10,
    verbose: bool = True
) -> TrazaAgente:
    """
    Agente de análisis de tecnología.

    TODO: implementa este agente usando ejecutar_agente_openai() como base,
    pero con:
    1. SYSTEM_TECH_ANALYST como prompt de sistema
    2. HERRAMIENTAS_EXTENDIDAS_OPENAI (incluye la nueva herramienta de comparación)
    3. Un pre-procesamiento: antes de ejecutar el bucle, añade al contexto
       la fecha actual (llamando a obtener_fecha_hora()) para que el agente
       sepa cuándo está trabajando

    Tip: puedes reutilizar ejecutar_agente_openai() directamente si modificas
    el SYSTEM_AGENTE, o puedes duplicar el bucle y personalizarlo.
    """
    # TODO: implementa el agente analista
    raise NotImplementedError("Implementa agente_analista()")


# ── Preguntas de prueba para el agente analista ───────────────────────────────
PREGUNTAS_ANALISTA = [
    "Should I use FastAPI or Django for building a REST API? Give me a recommendation.",
    "What is the current state of Python in AI/ML? What are the most recent developments?",
    "PostgreSQL has been around for decades. How old is it exactly and is it still actively developed?",
]

# Descomenta cuando hayas implementado agente_analista()
# for pregunta in PREGUNTAS_ANALISTA:
#     print(f"\n{'='*65}")
#     traza = agente_analista(pregunta, verbose=True)
#     print(f"\nINFORME:\n{traza.respuesta}")
#     traza.resumen()

### Extensiones opcionales

1. **Agente con memoria entre sesiones:** guarda el historial de conversación en un archivo JSON y cárgalo al iniciar. Permite hacer preguntas de seguimiento en una sesión posterior.

2. **Límite de coste dinámico:** añade un parámetro `max_coste_usd` al agente. Antes de cada llamada al LLM, calcula el coste acumulado y para si se supera el límite.

3. **Herramienta de RAG:** integra la colección `rag_docs` del notebook 07 como herramienta del agente. El agente podrá elegir entre buscar en Wikipedia, buscar en noticias, **o buscar en tu corpus privado**.

In [ ]:
# Extensión 3: esqueleto para integrar RAG como herramienta
# Descomenta y completa si tienes la colección rag_docs del notebook 07

# import chromadb
# chroma_cliente = chromadb.PersistentClient(path="./chroma_db")

# def buscar_documentacion_interna(query: str, k: int = 3) -> str:
#     """
#     Search the internal documentation corpus (FastAPI, REST, Pydantic, JWT, Docker, PostgreSQL).
#     Use this for technical documentation queries about these specific technologies.
#     """
#     try:
#         coleccion = chroma_cliente.get_collection("rag_docs")
#         # TODO: implementa el retrieval usando generar_embedding() del notebook 07
#         ...
#     except Exception as e:
#         return f"Internal docs not available: {e}"

# HERRAMIENTA_RAG = {
#     "type": "function",
#     "function": {
#         "name": "buscar_documentacion_interna",
#         "description": (
#             "Search the internal technical documentation corpus. "
#             "Topics covered: FastAPI, REST APIs, Pydantic, JWT, Docker, PostgreSQL. "
#             "Prefer this over Wikipedia for detailed technical queries on these topics."
#         ),
#         "parameters": {
#             "type": "object",
#             "properties": {
#                 "query": {"type": "string", "description": "Technical question or search term"},
#                 "k":     {"type": "integer", "description": "Number of results (1-5)", "default": 3}
#             },
#             "required": ["query"]
#         }
#     }
# }

---
## Autoevaluación de la semana

1. **¿Qué funcionó bien?**  
   ...

2. **¿Qué no entendiste del todo?**  
   ...

3. **¿Qué cambiarías la semana que viene?**  
   ...

4. **Tiempo real dedicado:**  
   ...

---
*Notebook 09 — Semana 9 del Plan de Formación AI Engineering*